In [1]:
import pennylane as qml
import numpy as np
from scipy.linalg import cossin
from scipy.stats import unitary_group
from rotoptsynth import (
    csd,
    de_mux,
    re_and_de_mux,
    balance_diagonal,
    asymmetric_decomp,
    two_qubit_flag_decomp,
    mottonen,
    one_qubit_flag_decomp,
    MultiplexedFlag,
    mux_ops,
    decompose_mux_single_qubit_flags,
)

def po_qsd(V: np.ndarray, wires: list) -> list:
    """
    Implements the Parameter-Optimal Quantum Shannon Decomposition (PO-QSD) skeleton.
    Assumes wires[0] is the target/multiplexed qubit (q_0) and wires[1:] are the controls (q_[1:]).
    """
    n = len(wires)
    assert V.shape == (2**n, 2**n)
    
    # Base Case
    if n == 2:
        return [qml.QubitUnitary(V, wires=wires)]

    with qml.QueuingManager.stop_recording():
        controls = wires[1:]
        target = wires[0]
        
        # 1. Cosine-Sine Decomposition 
        K00, K01, theta_y, K10, K11 = csd(V)
        M10, theta_Z_1, M11 = de_mux(K10, K11)
        M00, theta_Z_0, M01 = de_mux(K00, K01)

        # Create re-multiplexed matrix into which we absorb CY gates. Note that we change
        # the basis on the target qubit from Y to Z implicitly. We don't need to compute this
        # basis change, we just use `CZ` instead of `CY` and pass `theta_y` to the _Z_ angles of
        # MultiplexedFlag. The change back is then by passing the newly computed angles 
        # theta_y_new to `mottonen` with `axis="Y"`.
        M01_new, theta_y_new, M10_new = re_and_de_mux(M01, M10, theta_y, wires, side="both")
        
        F_11, Delta_11 = recursive_flag(M11, controls, n_b=2, partial_demux=True)
        F_10, Delta_10 = recursive_flag(M10_new * Delta_11, controls, n_b=2, partial_demux=True)
        F_01, Delta_01 = recursive_flag(M01_new * Delta_10, controls, n_b=2, partial_demux=True)
        
        # Recurse PO-QSD on the final modified M00 block
        C00 = po_qsd(M00 * Delta_01, controls)

        # First Mottonen decomposition (Z-rotations) 
        C_Z_1 = mottonen(theta_Z_1, controls, target, axis='Z', symmetrized="right")
        # 6. Second Mottonen decomposition (Y-rotations) 
        C_Y = mottonen(theta_y_new, controls, target, axis='Y')
        C_Z_0 = mottonen(theta_Z_0, controls, target, axis='Z', symmetrized="left")
        circuit = F_11 + C_Z_1 + F_10 + C_Y + F_01 + C_Z_0 + C00
        
    if qml.QueuingManager.recording():
        for op in circuit:
            qml.apply(op)
    # Combine all operations [cite: 230]
    return circuit

In [2]:
def recursive_flag(V: np.ndarray, wires: list, n_b: int = 2, partial_demux: bool = False) -> tuple[list, np.ndarray]:
    """
    Implements the recursive flag decomposition (Algorithm 1).
    
    Args:
        V (np.ndarray): A 2^n x 2^n complex unitary matrix.
        wires (list): The list of wires [q_0, q_1, ..., q_{n-1}].
        n_b (int): Base case threshold (1 or 2).
        
    Returns:
        tuple: (F, Delta) where:
            - F is a list of PennyLane operations (the flag circuit).
            - Delta is a 1D numpy array representing the extracted diagonal matrix.
    """
    # print(f"Calling `recursive_flag` with {V.shape=}, {wires=}, {n_b=}")
    n = len(wires)
    
    # Base cases
    if n_b == 1 and n == 1:
        return one_qubit_flag_decomp(V, wires)
    elif n_b == 2 and n == 2:
        return two_qubit_flag_decomp(V, wires)
        
    # 1. Cosine-Sine Decomposition
    K00, K01, theta_Y, K10, K11 = csd(V)
    
    controls = wires[1:]
    target = wires[0]
    
    if partial_demux:
        # Selective de-multiplexing branch
        M10, theta_Z_prime, M11 = de_mux(K10, K11)
        
        F11, Delta = recursive_flag(M11, controls, n_b=n_b, partial_demux=True)
        F11, Delta_11_mod = decompose_mux_single_qubit_flags(F11)
        Delta = Delta * Delta_11_mod
        
        assert np.allclose(M11, np.diag(Delta) @ qml.matrix(F11, wire_order=controls))
        
        F_Z = mottonen(theta_Z_prime, controls, target, axis="Z", symmetrized="right")
        # F_Z.append(qml.CY([controls[0], target]))

        # The following de-multiplexing only is done to enable symmetrized Mottonen it is undone later.
        M00, theta_Z_0, M01 = de_mux(K00, K01)
        M01_new, theta_Y_new, M10_new = re_and_de_mux(M01, M10, theta_Y, wires, side="left")
        
        F10, Delta_prime = recursive_flag(M10_new * Delta, controls, n_b=n_b, partial_demux=True)
        F10, Delta_10_mod = decompose_mux_single_qubit_flags(F10)
        Delta_prime = Delta_prime * Delta_10_mod

        # assert np.allclose(M10 * Delta, np.diag(Delta_prime) @ qml.matrix(F10, wire_order=controls))
        
        F_Y = mottonen(theta_Y_new, controls, target, axis="Y", symmetrized="right")
        
        F_top = F11 + F_Z + F10 + F_Y

        _cz = qml.CZ([controls[0], target]).matrix(wire_order=wires)
        re_mux_rhs = np.kron(np.eye(2), M00) @ qml.matrix(mottonen(theta_Z_0, controls, target, axis="Z"), wire_order=wires) @ np.kron(np.eye(2), M01_new) @ _cz
        K00 = re_mux_rhs[:2**(n-1),:2**(n-1)]
        K01 = re_mux_rhs[2**(n-1):,2**(n-1):]
        
    else:
        # Standard n_b = 1 recursive branch
        F10, Delta10 = recursive_flag(K10, controls, n_b=n_b, partial_demux=False)
        F11, Delta11 = recursive_flag(K11, controls, n_b=n_b, partial_demux=False)
        
        # Combine the diagonals and split them into a Z-rotation and a residual diagonal
        combined_Delta = np.concatenate([Delta10, Delta11])
        theta_Z, Delta_prime = balance_diagonal(combined_Delta, -1)
        
        F_A = MultiplexedFlag(theta_Z, theta_Y, controls+[target])
        
        # attach a multiplexer control to F10 and F11 based on the target qubit
        F_top = mux_ops(F10, F11, control=target) + [F_A]
        
    # Common continuation for K00 and K01 blocks
    F00, Delta00 = recursive_flag(K00 * Delta_prime, controls, n_b, partial_demux=False)    
    F01, Delta01 = recursive_flag(K01 * Delta_prime, controls, n_b, partial_demux=False)

    # mat = (
    #     qml.math.block_diag([K00, K01])
    #     @ qml.matrix(mottonen(0*theta_Y, theta_Y_new, controls+[target]), wire_order=wires)
    #     @ np.kron(np.eye(2), M10_new)
    #     @ qml.matrix(MultiplexedFlag(theta_Z_prime, 0*theta_Z_prime, controls+[target]), wire_order=wires)
    #     @ np.kron(np.eye(2), M11)
    # )
    # assert np.allclose(mat, V)
    
    # The final diagonal is the concatenation of the 0 and 1 branches
    Delta_out = np.concatenate([Delta00, Delta01])
    
    # Attach multiplexer controls to F00 and F01
    F_bottom = mux_ops(F00, F01, control=target)
    F_bottom, Delta_out_mod = decompose_mux_single_qubit_flags(F_bottom)
    Delta_out = Delta_out * Delta_out_mod
    
    return F_top + F_bottom, Delta_out

# To do

- [x] Symmetrized Mottonen decomposition
- [x] What about de_mux_flags??
- [x] Stray CNOTs?
- [ ] Modularized decomposition via PL decomp system instead?

In [3]:
n = 5

N = 2**n
dev = qml.device("default.qubit", wires=n)
wires = list(range(n))

gate_set = {"CZ", "RX", "RY", "RZ", "Hadamard", "S", "GlobalPhase", "X", "Y", "Z"}#, "MultiplexedFlag"}

@qml.transforms.combine_global_phases
@qml.decompose(gate_set=gate_set)
@qml.qnode(dev)
def node(V):
    po_qsd(V, wires)
    return qml.state()

# for _ in range(1000):
V = unitary_group.rvs(N)
ops = po_qsd(V, wires) # Just see if it runs

print(qml.draw(node, max_length=150, show_matrices=False)(V))
gates = qml.specs(node)(V)["resources"].gate_types
# print(gates)
print(f"Rotations: {sum([val for key, val in gates.items() if key in ["RX", "RY", "RZ"]])}")
print(f"CNOTs: {sum([val for key, val in gates.items() if key in ["CNOT", "CZ", "CY"]])}")
print(f"GlobalPhase: {'✅' if "GlobalPhase" in gates else '❌'}\n")
print(f"Paper says:\nRotations: {4**n-1}\nCNOTs: {4**n / 2 - 3/8*(n+2)*2**n + n -1 }")
mat = qml.matrix(po_qsd, wire_order=wires)(V, wires) # Compute matrix
assert np.allclose(mat, V)

3: ──MultiplexedFlag(M0,M1)─╭●──MultiplexedFlag(M4,M5)─╭●──MultiplexedFlag(M8,M9)───┤  
4: ──MultiplexedFlag(M2,M3)─╰Z──MultiplexedFlag(M6,M7)─╰Z──MultiplexedFlag(M10,M11)─┤  




3: ──MultiplexedFlag(M0,M1)─╭●──MultiplexedFlag(M4,M5)─╭●──MultiplexedFlag(M8,M9)───┤  
4: ──MultiplexedFlag(M2,M3)─╰Z──MultiplexedFlag(M6,M7)─╰Z──MultiplexedFlag(M10,M11)─┤  




2: ─╭MultiplexedFlag(M0,M1)─╭MultiplexedFlag(M2,M3)────╭MultiplexedFlag(M4,M5) ···
3: ─╰MultiplexedFlag(M0,M1)─│───────────────────────╭●─╰MultiplexedFlag(M4,M5) ···
4: ─────────────────────────╰MultiplexedFlag(M2,M3)─╰Z──────────────────────── ···

2: ··· ─╭MultiplexedFlag(M6,M7)────╭MultiplexedFlag(M8,M9)─╭MultiplexedFlag(M10,M11)─┤  
3: ··· ─│───────────────────────╭●─╰MultiplexedFlag(M8,M9)─│─────────────────────────┤  
4: ··· ─╰MultiplexedFlag(M6,M7)─╰Z─────────────────────────╰MultiplexedFlag(M10,M11)─┤  




2: ──RZ───────────────────────────╭X──RZ─╭X──RZ─╭X──RZ──RY───────────────────────╭X──RY─╭X──RY ···
3: ──RZ──RY─╭●──RZ──

AssertionError: can't merge a diagonal into a smaller MultiplexedFlag.

In [ ]:
print(4**4)